# IsoGrowingNCA Chameleon LPPN Demo

Self-contained Colab-style demo for testing whether Cartesian LPPN coordinates break equivariance for an isotropic growing NCA.

The notebook trains two LPPN models on `data/morphology_png/chameleon.png`:

- `coords`: normal LPPN/SIREN with local Cartesian coordinates.
- `no_coords`: control decoder with `disable_coords=True`.

It then compares `rollout(transform(seed))` against `transform(rollout(seed))` for exact rotations, reflections, and arbitrary interpolated rotations.

For the no-LPPN direct RGBA baseline, use `notebooks/isonca_chameleon_baseline.ipynb`.


In [ ]:
#@title Clone repository (Colab setup)
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/IvanLudvig/Cells2Pixels.git'
BRANCH = 'isotropic'
REPO_DIR = 'Cells2Pixels'

if not Path('train.py').exists():
    if not Path(REPO_DIR).exists():
        cmd = ['git', 'clone']
        if BRANCH:
            cmd += ['--branch', BRANCH]
        cmd += [REPO_URL, REPO_DIR]
        subprocess.run(cmd, check=True)
    os.chdir(REPO_DIR)

print('cwd:', Path.cwd())


In [ ]:
#@title Imports and device setup
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import clear_output, display
from PIL import Image

from losses.loss import Loss
from models.isonca import IsoGrowingNCA
from models.siren import Siren
from training.common import (
    make_grad_scaler,
    normalize_model_grads,
    optimizer_scheduler_step,
    set_seed,
)
from utils.misc import autocast_context
from utils.render import Renderer2D

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this notebook. Enable a GPU runtime in Colab.')

device = torch.device('cuda:0')
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
#@title Download data
TARGET_IMAGE_PATH = Path('data/morphology_png/chameleon.png')
if not TARGET_IMAGE_PATH.exists():
    print('Dataset missing; running scripts/download_data.py. This may take a while.')
    subprocess.run([sys.executable, 'scripts/download_data.py'], check=True)

if not TARGET_IMAGE_PATH.exists():
    raise FileNotFoundError(f'Missing target image: {TARGET_IMAGE_PATH}')

img = Image.open(TARGET_IMAGE_PATH).convert('RGBA')
print('Target:', TARGET_IMAGE_PATH, '| size:', img.size)
display(img.resize((360, round(360 * img.height / img.width))))


In [ ]:
#@title Experiment settings
SEED = 43
PRECISION = torch.float16
OUTPUT_TYPE = 's'

# Lighter than the full YAML configs so both LPPN variants can run in a notebook.
EPOCHS = 1500  #@param {type: 'integer'}
BATCH_SIZE = 4  #@param {type: 'integer'}
POOL_SIZE = 256  #@param {type: 'integer'}
STEP_RANGE = (32, 96)
INJECT_SEED_INTERVAL = 32
SUMMARY_INTERVAL = 100

CHANNELS = 32
FC_DIM = 256
UPDATE_PROB = 0.5
SCALE_FACTOR = 8
IMAGE_SIZE = (512, 512)
PADDING = (128, 128)
LPIPS_WEIGHT = 0.0  # Set to 1.0 for higher-quality but slower training.

EVAL_STEPS = 512
ANGLES = [30.0, 45.0, 60.0, 120.0]
RUN_VARIANTS = ['coords', 'no_coords']

set_seed(SEED)


In [ ]:
#@title Build/train helpers
def make_loss():
    return Loss(
        overflow_loss_weight=100.0,
        image_loss_weight=1.0,
        image_loss_kwargs=dict(
            target_path=str(TARGET_IMAGE_PATH),
            image_size=list(IMAGE_SIZE),
            padding=list(PADDING),
            l2_weight=1.0,
            l1_weight=1.0,
            lpips_weight=LPIPS_WEIGHT,
            premultiply_alpha=True,
            device=str(device),
        ),
    )


def build_variant(name):
    model = IsoGrowingNCA(
        channels=CHANNELS,
        fc_dim=FC_DIM,
        update_prob=UPDATE_PROB,
        device=device,
        precision=PRECISION,
    ).to(device)

    disable_coords = name == 'no_coords'
    nca_output_dim = model.channels
    if OUTPUT_TYPE == 'z':
        nca_output_dim *= model.perception_kernels

    siren = Siren(
        in_features=nca_output_dim,
        coord_dim=2,
        out_features=4,
        hidden_features=64,
        hidden_layers=2,
        outermost_linear=True,
        fx='linear',
        activation='sin',
        num_frequencies=1,
        first_omega_0=10.0,
        hidden_omega_0=10.0,
        disable=False,
        disable_coords=disable_coords,
    ).to(device)

    loss_fn = make_loss()
    renderer = Renderer2D(scale_factor=SCALE_FACTOR, padding='circular', fs_shader='vanilla', precision=PRECISION)
    grid_size = loss_fn.loss_mapper['image'].grid_size
    grid_size = (grid_size[0] // renderer.scale_factor, grid_size[1] // renderer.scale_factor)
    return model, siren, loss_fn, renderer, grid_size


@torch.no_grad()
def render_rgba(model, siren, renderer, state, perception=None):
    x_render = state if OUTPUT_TYPE == 's' or perception is None else perception
    image = renderer.render(
        x_render.to(torch.float32).permute(0, 2, 3, 1),
        siren,
        None,
        fs_shader='vanilla',
        hard_clamp=True,
    )
    x_up = torch.nn.functional.interpolate(state.to(torch.float32), scale_factor=renderer.scale_factor, mode='bilinear')
    mask = model.get_living_mask(x_up).float().permute(0, 2, 3, 1)
    return image.to(torch.float32) * mask


def composite_rgba(image):
    rgb = image[..., :3]
    alpha = image[..., 3:4]
    return rgb * alpha + (1.0 - alpha)


In [ ]:
#@title Train variants
def train_variant(name):
    set_seed(SEED)
    model, siren, loss_fn, renderer, grid_size = build_variant(name)
    with torch.no_grad():
        pool = model.seed(POOL_SIZE, grid_size[0], grid_size[1])

    optimizer = torch.optim.Adam(list(model.parameters()) + list(siren.parameters()), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[max(1, EPOCHS // 3), max(2, 2 * EPOCHS // 3)],
        gamma=0.3,
    )
    scaler = make_grad_scaler(device, PRECISION)
    loss_history = []

    model.train()
    siren.train()
    for epoch in range(EPOCHS + 1):
        with torch.no_grad():
            batch_idx = np.random.choice(len(pool), BATCH_SIZE, replace=False)
            x = pool[batch_idx]
            if epoch % INJECT_SEED_INTERVAL == 0:
                x[:1] = model.seed(1, grid_size[0], grid_size[1])

        step_n = np.random.randint(*STEP_RANGE)
        z = None
        with autocast_context(device, PRECISION):
            for _ in range(step_n):
                x, z = model(x)

        rendered = render_rgba(model, siren, renderer, x, z).permute(0, 3, 1, 2)
        x_up = torch.nn.functional.interpolate(x.to(torch.float32), scale_factor=renderer.scale_factor, mode='bilinear')
        input_dict = {
            'generated_images': rendered,
            'alpha': x_up[:, 3:4],
            'nca_state': x,
        }
        return_summary = epoch % SUMMARY_INTERVAL == 0
        loss, loss_log, summary = loss_fn(input_dict, return_summary=return_summary)
        if not torch.isfinite(loss):
            raise ValueError(f'{name} loss is NaN or Inf at epoch {epoch}')

        if PRECISION == torch.float16:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        with torch.no_grad():
            normalize_model_grads(model)
            normalize_model_grads(siren)
            optimizer_scheduler_step(optimizer, scheduler, scaler, PRECISION)
            optimizer.zero_grad()
            pool[batch_idx] = x

        loss_history.append(loss.item())
        if return_summary:
            clear_output(wait=True)
            fig, axes = plt.subplots(2, 1, figsize=(7, 10))
            axes[0].imshow(summary['image-images'])
            axes[0].set_title(f'{name}: target/generated/difference - epoch {epoch}')
            axes[0].axis('off')
            axes[1].plot(loss_history, '.', alpha=0.25)
            axes[1].set_yscale('log')
            axes[1].set_title(f'loss = {loss_history[-1]:.4f}')
            plt.tight_layout()
            plt.show()
            print(f'{name}: epoch {epoch}/{EPOCHS} | loss: {loss_history[-1]:.6g}')

    out_dir = Path('notebook_runs') / f'isonca_chameleon_{name}'
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out_dir / 'model.pth')
    torch.save(siren.state_dict(), out_dir / 'siren.pth')
    print('saved:', out_dir)
    return {
        'name': name,
        'model': model.eval(),
        'siren': siren.eval(),
        'loss_fn': loss_fn,
        'renderer': renderer,
        'grid_size': grid_size,
        'loss_history': loss_history,
        'path': out_dir,
    }


variants = {}
for variant_name in RUN_VARIANTS:
    variants[variant_name] = train_variant(variant_name)


In [ ]:
#@title Equivariance metrics
def rollout(model, x, steps):
    z = None
    with autocast_context(device, PRECISION):
        for _ in range(steps):
            x, z = model(x)
    return x, z


def rel_l2(a, b):
    return torch.linalg.vector_norm((a - b).reshape(a.shape[0], -1), dim=1) / (
        torch.linalg.vector_norm(b.reshape(b.shape[0], -1), dim=1) + 1e-8
    )


def affine_rotate_nchw(x, angle_deg):
    angle = torch.tensor(angle_deg * torch.pi / 180.0, device=x.device, dtype=x.dtype)
    c = torch.cos(angle)
    s = torch.sin(angle)
    theta = torch.zeros(x.shape[0], 2, 3, device=x.device, dtype=x.dtype)
    theta[:, 0, 0] = c
    theta[:, 0, 1] = -s
    theta[:, 1, 0] = s
    theta[:, 1, 1] = c
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, mode='bilinear', padding_mode='zeros', align_corners=False)


def affine_rotate_nhwc(image, angle_deg):
    x = image.permute(0, 3, 1, 2)
    x = affine_rotate_nchw(x, angle_deg)
    return x.permute(0, 2, 3, 1)


@torch.no_grad()
def evaluate_variant(variant, steps=EVAL_STEPS, angles=ANGLES):
    model = variant['model']
    siren = variant['siren']
    renderer = variant['renderer']
    old_update_prob = model.update_prob
    model.update_prob = 1.0
    rows = []
    visuals = {}
    try:
        h, w = variant['grid_size']
        x0 = model.seed(1, h, w)
        x_base, z_base = rollout(model, x0, steps)
        image_base = render_rgba(model, siren, renderer, x_base, z_base)

        def add_result(kind, label, state, expected_state, image, expected_image):
            err = (image - expected_image).abs()
            rows.append({
                'kind': kind,
                'transform': label,
                'state_mae': (state - expected_state).abs().mean().item(),
                'state_rel_l2': rel_l2(state, expected_state).mean().item(),
                'render_mae': err.mean().item(),
                'render_rel_l2': rel_l2(image, expected_image).mean().item(),
            })
            visuals[label] = {
                'base': image_base.detach().cpu(),
                'transformed_seed': image.detach().cpu(),
                'transformed_output': expected_image.detach().cpu(),
                'error': err.detach().cpu(),
            }

        for k in (1, 2, 3):
            x_t0 = torch.rot90(x0, k, dims=(-2, -1))
            x_t, z_t = rollout(model, x_t0, steps)
            image_t = render_rgba(model, siren, renderer, x_t, z_t)
            add_result('exact_rotation', f'rot{k * 90}', x_t, torch.rot90(x_base, k, dims=(-2, -1)), image_t, torch.rot90(image_base, k, dims=(-3, -2)))

        for label, dims in (('flip_x', (-1,)), ('flip_y', (-2,))):
            x_t0 = torch.flip(x0, dims=dims)
            x_t, z_t = rollout(model, x_t0, steps)
            image_t = render_rgba(model, siren, renderer, x_t, z_t)
            image_dims = tuple(d - 1 for d in dims)
            add_result('reflection', label, x_t, torch.flip(x_base, dims=dims), image_t, torch.flip(image_base, dims=image_dims))

        for angle in angles:
            x_t0 = affine_rotate_nchw(x0, angle)
            x_t, z_t = rollout(model, x_t0, steps)
            image_t = render_rgba(model, siren, renderer, x_t, z_t)
            add_result('interpolated_rotation', f'rot{angle:g}', x_t, affine_rotate_nchw(x_base, angle), image_t, affine_rotate_nhwc(image_base, angle))
    finally:
        model.update_prob = old_update_prob
    return rows, visuals


results = {}
visuals = {}
for name, variant in variants.items():
    rows, vis = evaluate_variant(variant)
    results[name] = rows
    visuals[name] = vis
    print()
    print(name)
    print('kind | transform | state_mae | state_rel_l2 | render_mae | render_rel_l2')
    for row in rows:
        print(f"{row['kind']} | {row['transform']} | {row['state_mae']:.3e} | {row['state_rel_l2']:.3e} | {row['render_mae']:.3e} | {row['render_rel_l2']:.3e}")


In [ ]:
#@title Visual comparison grids
def show_grid(variant_name, transform='rot45'):
    item = visuals[variant_name][transform]
    titles = ['original render', 'rollout(transform(seed))', 'transform(rollout(seed))', 'absolute error']
    tensors = [item['base'], item['transformed_seed'], item['transformed_output'], item['error']]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for ax, title, tensor in zip(axes, titles, tensors):
        img = tensor[0].numpy()
        if title == 'absolute error':
            ax.imshow(img.mean(axis=-1), cmap='magma')
        else:
            ax.imshow(np.clip(composite_rgba(img), 0.0, 1.0))
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'{variant_name}: {transform}')
    plt.tight_layout()
    plt.show()


for name in visuals:
    show_grid(name, 'rot45')


In [ ]:
# Try other transforms after the previous cell runs:
# show_grid('coords', 'flip_x')
# show_grid('coords', 'rot90')
# show_grid('no_coords', 'rot60')
